In [4]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class BossDuelEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self):
        super().__init__()
        # 游戏参数
        self.max_health = 100
        self.boss_health_max = 200
        self.max_energy = 100
        self.max_stamina = 100
        self.max_steps = 50

        # 动作空间：4个离散动作
        self.action_space = spaces.Discrete(4)
        # 观测空间：8维向量，范围0~1
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(8,), dtype=np.float32)

        # Boss 动作概率（轻击、重击、蓄力）
        self.boss_probs = [0.4, 0.5, 0.1]

        # 伤害值
        self.agent_normal_damage = 10
        self.agent_ult_damage = 30
        self.boss_light_damage = 10
        self.boss_heavy_damage = 25

        # 动作消耗/收益
        self.stamina_cost_dodge = 20
        self.energy_gain_normal = 10
        self.energy_gain_dodge = 5
        self.energy_gain_block = 15

        self.reset()

    def _get_state_with_boss_action(self, boss_action):
        boss_onehot = np.zeros(3)
        boss_onehot[boss_action] = 1
        state = np.concatenate([
            boss_onehot,
            [self.agent_health / self.max_health],
            [self.boss_health / self.boss_health_max],
            [self.agent_energy / self.max_energy],
            [self.agent_stamina / self.max_stamina],
            [self.boss_charge]
        ])
        return state.astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)  # 必须调用，用于设置随机种子
        self.agent_health = self.max_health
        self.agent_energy = 0
        self.agent_stamina = self.max_stamina
        self.boss_charge = 0
        self.boss_health = self.boss_health_max
        self.steps = 0
        boss_action = np.random.choice([0, 1, 2], p=self.boss_probs)
        self.state = self._get_state_with_boss_action(boss_action)
        info = {}  # 可以放一些调试信息
        return self.state, info

    def step(self, action):
        self.steps += 1

        boss_action = np.argmax(self.state[:3])
        is_boss_charged = (self.boss_charge == 1)
        self.boss_charge = 0

        reward = 0
        agent_attack_damage = 0
        agent_taken_damage = 0

        # ========== 先处理 Boss 动作 ==========
        if boss_action == 2:  # Boss 蓄力
            self.boss_charge = 1
            # 打断判断：大招且能量≥一半
            if action == 2 and self.agent_energy >= self.max_energy / 2 - 1e-5:
                self.boss_charge = 0
                reward += 10

        # ========== 处理 Agent 动作 ==========
        # 大招
        if action == 2:
            if self.agent_energy >= self.max_energy / 2 - 1e-5:
                damage = self.agent_ult_damage
                if np.random.rand() < 0.2:
                    damage = int(damage * 1.5)
                agent_attack_damage = damage
                reward += damage * 0.1
                reward += 5
                self.agent_energy = max(0, self.agent_energy - self.max_energy / 2)
            else:
                # 能量不足，转为普攻
                action = 3

        # 普攻（主动或转来）
        if action == 3:
            damage = self.agent_normal_damage + np.random.randint(-2, 3)
            agent_attack_damage = damage
            reward += damage * 0.1
            self.agent_energy = min(self.max_energy, self.agent_energy + self.energy_gain_normal)

        # 应用 Agent 伤害
        self.boss_health -= agent_attack_damage

        # ========== 处理 Boss 攻击 ==========
        if boss_action != 2:
            base_damage = self.boss_heavy_damage if boss_action == 0 else self.boss_light_damage
            if is_boss_charged:
                base_damage = int(base_damage * 2)

            if action == 0:  # 格挡
                if is_boss_charged:
                    actual_damage = base_damage
                else:
                    actual_damage = int(base_damage * 0.2)
                self.agent_energy = min(self.max_energy, self.agent_energy + self.energy_gain_block)

            elif action == 1:  # 闪避
                if self.agent_stamina >= self.stamina_cost_dodge:
                    self.agent_stamina -= self.stamina_cost_dodge
                    if is_boss_charged:
                        actual_damage = base_damage
                    else:
                        actual_damage = 0
                        reward += 1
                    self.agent_energy = min(self.max_energy, self.agent_energy + self.energy_gain_dodge)
                else:
                    actual_damage = base_damage
            else:
                actual_damage = base_damage

            agent_taken_damage = actual_damage
            self.agent_health -= agent_taken_damage
            reward -= agent_taken_damage * 0.2

        # 体力自然恢复
        self.agent_stamina = min(self.max_stamina, self.agent_stamina + 5)

        # 边界限制
        self.agent_health = max(0, min(self.max_health, self.agent_health))
        self.boss_health = max(0, min(self.boss_health_max, self.boss_health))
        self.agent_energy = max(0, min(self.max_energy, self.agent_energy))
        self.agent_stamina = max(0, min(self.max_stamina, self.agent_stamina))

        # 终止判断（gymnasium 用 terminated 和 truncated）
        terminated = False
        truncated = False
        if self.agent_health <= 0 or self.boss_health <= 0:
            terminated = True
        elif self.steps >= self.max_steps:
            truncated = True

        # 奖励处理
        if self.agent_health <= 0:
            reward -= 200
        elif self.boss_health <= 0:
            reward += 200
        elif truncated:
            reward += (self.agent_health - self.boss_health) / 10

        # 生成下一状态
        if not (terminated or truncated):
            next_boss_action = np.random.choice([0, 1, 2], p=self.boss_probs)
            next_state = self._get_state_with_boss_action(next_boss_action)
        else:
            next_state = np.zeros(8, dtype=np.float32)

        self.state = next_state

        info = {
            'boss_action': boss_action,
            'agent_action': action,
            'damage_dealt': agent_attack_damage,
            'damage_taken': agent_taken_damage
        }

        return next_state, reward, terminated, truncated, info

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold


# 注册环境（可选，也可以直接传入环境类）
# 为了使用 make_vec_env，我们直接传入环境类

# 创建向量化环境（可并行多个环境，加速训练）
# 这里先用单环境，方便观察
env = BossDuelEnv()

# 定义模型
# MlpPolicy 适用于向量输入（我们的状态是 8 维向量）
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,          # 每次更新前收集的步数
    batch_size=64,
    n_epochs=10,           # 每次更新时对样本重复使用的次数
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,         # 探索系数，可适当提高
    tensorboard_log="./ppo_boss_duel_tensorboard/"
)

# 可选：添加评估回调，当平均奖励超过阈值时停止训练
# 这里阈值设为 150（因为 Boss 死亡奖励 200，胜局总奖励会接近 200）
eval_callback = EvalCallback(
    env,
    eval_freq=50000,
    deterministic=True,
    render=False,
    verbose=1
)
model = PPO.load("ppo_second.zip")
model.set_env(env)
model.learn(
    total_timesteps=12000000,          # 新增加的步数
    callback=eval_callback,
    reset_num_timesteps=False        # 保持计数器延续
)
# 保存最终模型
model.save("ppo_treth")

print("训练完成，模型已保存为 ppo_treth.zip")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./ppo_boss_duel_tensorboard/PPO_6


c:\Users\MR\anaconda3\envs\Herta\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 45.3     |
|    ep_rew_mean     | 14       |
| time/              |          |
|    fps             | 333      |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 3004416  |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 44.1         |
|    ep_rew_mean          | 5.11         |
| time/                   |              |
|    fps                  | 288          |
|    iterations           | 2            |
|    time_elapsed         | 14           |
|    total_timesteps      | 3006464      |
| train/                  |              |
|    approx_kl            | 0.0010848048 |
|    clip_fraction        | 0.00825      |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.0626      |
|    explained_variance   | 0.352        |
|    learning_r

c:\Users\MR\anaconda3\envs\Herta\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=3052368, episode_reward=3.30 +/- 108.99
Episode length: 46.60 +/- 6.80
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 46.6         |
|    mean_reward          | 3.3          |
| time/                   |              |
|    total_timesteps      | 3052368      |
| train/                  |              |
|    approx_kl            | 0.0010554426 |
|    clip_fraction        | 0.0085       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.0701      |
|    explained_variance   | 0.505        |
|    learning_rate        | 0.0003       |
|    loss                 | 1.55e+03     |
|    n_updates            | 14900        |
|    policy_gradient_loss | 0.000241     |
|    value_loss           | 2.78e+03     |
------------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 44.6     |
|    ep_rew_

: 